# verify03: モデルA（痛み/しびれ/振る舞い） vs モデルB（遷移）だけを比較

**方針**：トリアージ・遷移のための関数**以外は `HeadacheBERT_painful_Finetuning.ipynb` のコードをそのまま使う**
（`set_seed` / `remove_stopwords` / `get_tokenizer` / `PainTextDataset` / `build_model` /
`validate_labels` / `validate_token_ids` / `train_one_fold` / metrics / `StratifiedKFold` を流用）。

| モデル | 学習 | 入力 |
|---|---|---|
| **モデルA（痛み/しびれ/振る舞い）** | 各ノード専用の単体分類器（painfulの `train_one_fold` をそのまま使用・162例×3本） | 採用ペア（質問+回答・ノード質問なし） |
| **B. 遷移BERT** | 3ノードを1本で共有学習（486例） | 質問 + 採用ペア |

**新規に書くのは「トリアージ/遷移」の部分だけ**：決定木 `branch_table`、`train_transition_fold`（共有学習）、
`predict_triage`（決定木トラバーサル）。

出力：①ノード別 accuracy/precision/recall/F1（painful と同じ指標）、②最終トリアージ accuracy/F1。

# 1. セットアップ（GPUは git clone / ローカルはそのまま）

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/enenen13/Emergency_task'
REPO_NAME = 'Emergency_task'
REPO_BRANCH = 'feature/headache-ablation-notebook'
IN_COLAB = 'google.colab' in sys.modules


def _find_repo_root(start):
    d = os.path.abspath(start)
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'dataset', 'input_pairs.csv')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


REPO_DIR = _find_repo_root(os.getcwd())
cloned = False
if REPO_DIR is None:
    if not os.path.isdir(REPO_NAME):
        print(f'git clone -b {REPO_BRANCH} {REPO_URL} ...')
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=True)
        cloned = True
    REPO_DIR = os.path.abspath(REPO_NAME)
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR)

if IN_COLAB or cloned:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers', 'sentencepiece', 'fugashi', 'unidic-lite',
                    'accelerate', 'pyyaml'], check=True)

CSV_PATH = os.path.join(REPO_DIR, 'dataset', 'input_pairs.csv')
YAML_PATH = os.path.join(REPO_DIR, 'transition_diagram', 'protocol.yaml')
OUT_DIR = os.path.join(REPO_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)
print('CSV :', CSV_PATH, '(exists:', os.path.exists(CSV_PATH), ')')

# 2. インポート & 乱数シード（painful 流用）

In [ ]:
import time
import random
from typing import List, Dict, Optional

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)

import fugashi

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用デバイス:', DEVICE)

SEED = 42


def set_seed(seed: int = SEED):
    """再現性確保のため、各種ライブラリの乱数シードを固定する。（painful 流用）"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## 2.1 実験設定（A/B 共通）

painful の設定に合わせる。`MAX_LENGTH=512`。best(run_052) は lr5e-5/epoch10/batch8/SWなし。
（重ければ MAX_LENGTH や NUM_EPOCHS を下げる）

In [ ]:
MODEL_NAME = 'cl-tohoku/bert-base-japanese-v3'
MAX_LENGTH = 512
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
BATCH_SIZE = 8
N_FOLDS = 5
USE_STOPWORDS = False
NUM_LABELS = 3

print(f'MODEL={MODEL_NAME} MAX_LENGTH={MAX_LENGTH} lr={LEARNING_RATE} '
      f'epochs={NUM_EPOCHS} batch={BATCH_SIZE} folds={N_FOLDS} stopwords={USE_STOPWORDS}')

# 3. データ読み込み・前処理（painful 流用）＋ 決定木（遷移=新規）

In [ ]:
df = pd.read_csv(CSV_PATH)
print('rows:', len(df), '/ patients:', df['id'].nunique())
assert NUM_LABELS == 3

# ===== ここから「遷移/トリアージ」用の新規部分：protocol.yaml から決定木を読む =====
import yaml
_proto = yaml.safe_load(open(YAML_PATH, encoding='utf-8'))
_h = next(p for p in _proto['protocols'] if p['id'] == 'headache')
fallback_triage = _h['fallback']['if_all_symptom_questions_negative']


def _is_branch(n):
    return 'choices' in n and not n.get('metadata_only', False)


def _parse_choice(c):
    if c.get('triage'):
        return {'action': 'terminal', 'triage': c['triage']}
    if c.get('next'):
        return {'action': 'next', 'next_id': c['next']}
    return {'action': 'fallback', 'triage': fallback_triage}


branch_table = [{'id': n['id'], 'question': n['question'],
                 'choices': [_parse_choice(c) for c in n['choices']]}
                for n in _h['nodes'] if _is_branch(n)]
branch_ids = {b['id'] for b in branch_table}
triage_decode = {0: 'R3', 1: 'R2', 2: 'Y2'}

NODES = [
    {'key': '痛み',   'adopt': '採用ペア_ひし形_全通り_頭痛_1', 'label': '痛み',   'bid': 'headache_sudden_severe'},
    {'key': 'しびれ', 'adopt': '採用ペア_ひし形_全通り_頭痛_2', 'label': 'しびれ', 'bid': 'headache_numbness_paralysis'},
    {'key': '振る舞い', 'adopt': '採用ペア_ひし形_全通り_頭痛_3', 'label': '振る舞い', 'bid': 'headache_abnormal_behavior'},
]
_qmap = {b['id']: b['question'] for b in branch_table}
for n in NODES:
    n['question'] = _qmap[n['bid']]
print('branch_table:', [b['id'] for b in branch_table], '| fallback:', fallback_triage)

# ===== painful 流用：ストップワード除去 =====
_tagger = fugashi.Tagger()
STOPWORD_EXTRA_WORDS = set([
    'の', 'は', 'を', 'に', 'が', 'で', 'と', 'も', 'から', 'より',
    'へ', 'や', 'など', 'ので', 'けど', 'けれど', '、', '。', 'です', 'ます'
])


def remove_stopwords(text: str) -> str:
    """fugashiで形態素解析し、STOPWORD_EXTRA_WORDS の表層形を除去（painful 流用）。"""
    tokens = []
    for word in _tagger(text):
        if word.surface in STOPWORD_EXTRA_WORDS:
            continue
        tokens.append(word.surface)
    return ''.join(tokens)


# painful の input_filtered_function を「ノード別＋全患者対応＋質問付与」に一般化したもの
def make_node_table(adopt_col, label_col, use_question=False, question=''):
    rows = []
    for pid, g in df.groupby('id'):
        adopted = g[g[adopt_col] == True]
        text = ' '.join(adopted['ペア'].astype(str).tolist()) if len(adopted) else '(発話なし)'
        if USE_STOPWORDS:
            text = remove_stopwords(text)
        if use_question:
            text = question + ' ' + text
        rows.append({'id': pid, 'text': text, 'label': int(g[label_col].iloc[0])})
    return pd.DataFrame(rows).set_index('id')

# 4. 学習・評価関数（painful の `train_one_fold` などをそのまま流用）

In [ ]:
_tokenizer_cache: Dict[str, 'AutoTokenizer'] = {}


def get_tokenizer(model_name: str):
    """モデル名ごとにTokenizerをキャッシュして再利用する。（painful 流用）"""
    if model_name not in _tokenizer_cache:
        _tokenizer_cache[model_name] = AutoTokenizer.from_pretrained(model_name)
    return _tokenizer_cache[model_name]


class PainTextDataset(Dataset):
    """テキストとラベルを受け取り、tokenizerでエンコードするDataset。（painful 流用）"""

    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = MAX_LENGTH):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoded = self.tokenizer(text, truncation=True, max_length=self.max_length,
                                 padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item


def build_model(model_name: str, tokenizer, num_labels: int = NUM_LABELS,
                attn_implementation: Optional[str] = None):
    """分類用モデルを新規ロード。語彙数ズレは resize で補正。（painful 流用）"""
    model_kwargs = {'num_labels': num_labels}
    if attn_implementation is not None:
        model_kwargs['attn_implementation'] = attn_implementation
    model = AutoModelForSequenceClassification.from_pretrained(model_name, **model_kwargs)
    if len(tokenizer) != model.config.vocab_size:
        print(f'  [警告] {model_name}: 語彙数{len(tokenizer)} != vocab_size{model.config.vocab_size} → resize')
        model.resize_token_embeddings(len(tokenizer))
    return model.to(DEVICE)


def validate_labels(labels: List[int], num_labels: int = NUM_LABELS):
    arr = np.array(labels)
    if arr.min() < 0 or arr.max() > (num_labels - 1):
        bad = sorted(set(arr[(arr < 0) | (arr > (num_labels - 1))].tolist()))
        raise ValueError(f'ラベルが範囲外 [0,{num_labels - 1}]: {bad}')


def validate_token_ids(input_ids: 'torch.Tensor', vocab_size: int):
    max_id, min_id = int(input_ids.max().item()), int(input_ids.min().item())
    if max_id >= vocab_size or min_id < 0:
        raise ValueError(f'トークンIDがvocab({vocab_size})の範囲外: [{min_id},{max_id}]')


def train_one_fold(model_name, train_texts, train_labels, test_texts, test_labels,
                   learning_rate, num_epochs, batch_size, max_length=MAX_LENGTH, seed=SEED):
    """1設定・1foldの学習と評価。（painful 流用そのまま）"""
    set_seed(seed)
    validate_labels(train_labels)
    validate_labels(test_labels)
    tokenizer = get_tokenizer(model_name)
    model = build_model(model_name, tokenizer)
    vocab_size = model.get_input_embeddings().num_embeddings
    train_dataset = PainTextDataset(train_texts, train_labels, tokenizer, max_length)
    test_dataset = PainTextDataset(test_texts, test_labels, tokenizer, max_length)
    validate_token_ids(train_dataset[0]['input_ids'], vocab_size)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    train_loss_history = []
    start_time = time.time()
    model.train()
    for epoch in range(num_epochs):
        epoch_losses = []
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        train_loss_history.append(float(np.mean(epoch_losses)))
    train_time_sec = time.time() - start_time
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            labels = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    res = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision_macro': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall_macro': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'train_time_sec': train_time_sec, 'train_loss_history': train_loss_history,
        'y_true': all_labels, 'y_pred': all_preds,
    }
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res


print('painful流用の train_one_fold などを定義しました。')

# 5. 遷移・トリアージ用の関数（ここだけ新規）

- `train_transition_fold`：3ノードを**1本で共有学習**し、各ノードの test を予測（painfulの学習ループを流用）。
- `predict_triage`：ノード予測（はい/いいえ/不明）で決定木を辿り最終トリアージを返す。

In [ ]:
def train_transition_fold(train_texts, train_labels, test_texts_by_node,
                          learning_rate, num_epochs, batch_size, max_length=MAX_LENGTH, seed=SEED):
    """B（遷移）：3ノードまとめて1本学習 → ノードごとに予測。
    学習ループは painful の train_one_fold と同じ作り。"""
    set_seed(seed)
    validate_labels(train_labels)
    tokenizer = get_tokenizer(MODEL_NAME)
    model = build_model(MODEL_NAME, tokenizer)
    train_loader = DataLoader(PainTextDataset(train_texts, train_labels, tokenizer, max_length),
                              batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    model.train()
    for epoch in range(num_epochs):
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch)
            outputs.loss.backward()
            optimizer.step()
    model.eval()
    preds_by_node = {}
    with torch.no_grad():
        for node_key, texts in test_texts_by_node.items():
            ds = PainTextDataset(texts, [0] * len(texts), tokenizer, max_length)
            yp = []
            for batch in DataLoader(ds, batch_size=batch_size, shuffle=False):
                batch.pop('labels')
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                yp += torch.argmax(model(**batch).logits, dim=-1).cpu().numpy().tolist()
            preds_by_node[node_key] = yp
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return preds_by_node


def predict_triage(pred_idx_by_bid):
    """ノード予測(0/1/2)で決定木を辿り、最終トリアージを返す。"""
    current = branch_table[0]['id']
    while True:
        b = next(x for x in branch_table if x['id'] == current)
        ch = b['choices'][pred_idx_by_bid[b['id']]]
        if ch['action'] in ('terminal', 'fallback'):
            return ch['triage']
        if ch.get('next_id') in branch_ids:
            current = ch['next_id']
            continue
        return fallback_triage


print('遷移/トリアージ関数（train_transition_fold, predict_triage）を定義。')

# 6. fold分割（患者単位5-fold・A/B共通）＋ 正解

In [ ]:
patients = np.array(sorted(df['id'].unique()))
triage_strat = np.array([int(df[df['id'] == p]['トリアージ'].iloc[0]) for p in patients])
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = [(patients[tr], patients[te]) for tr, te in skf.split(patients, triage_strat)]
print(f'{N_FOLDS}-fold。test患者数:', [len(te) for _, te in FOLDS])

true_node = {n['key']: {p: int(df[df['id'] == p][n['label']].iloc[0]) for p in patients} for n in NODES}
true_triage = {p: triage_decode[int(df[df['id'] == p]['トリアージ'].iloc[0])] for p in patients}

# 7. モデルA：各ノード単体（painfull の train_one_fold をそのまま使用）

In [ ]:
tabsA = {n['key']: make_node_table(n['adopt'], n['label'], use_question=False) for n in NODES}
predA = {n['key']: {} for n in NODES}
metricsA = {n['key']: [] for n in NODES}

for n in NODES:
    tab = tabsA[n['key']]
    for i, (tr_ids, te_ids) in enumerate(FOLDS):
        res = train_one_fold(
            MODEL_NAME,
            tab.loc[tr_ids, 'text'].tolist(), tab.loc[tr_ids, 'label'].tolist(),
            tab.loc[te_ids, 'text'].tolist(), tab.loc[te_ids, 'label'].tolist(),
            LEARNING_RATE, NUM_EPOCHS, BATCH_SIZE, MAX_LENGTH)
        for pid, p in zip(te_ids, res['y_pred']):
            predA[n['key']][pid] = p
        metricsA[n['key']].append(res)
        print(f"  A[{n['key']}] fold{i}: acc={res['accuracy']:.3f} f1={res['f1_macro']:.3f}")
print('モデルA 完了')

# 8. モデルB：遷移（3ノード共有1本・新規の train_transition_fold）

In [ ]:
tabsB = {n['key']: make_node_table(n['adopt'], n['label'], use_question=True, question=n['question'])
         for n in NODES}
predB = {n['key']: {} for n in NODES}

for i, (tr_ids, te_ids) in enumerate(FOLDS):
    train_texts, train_labels = [], []
    for n in NODES:
        t = tabsB[n['key']]
        train_texts += t.loc[tr_ids, 'text'].tolist()
        train_labels += t.loc[tr_ids, 'label'].tolist()
    test_by_node = {n['key']: tabsB[n['key']].loc[te_ids, 'text'].tolist() for n in NODES}
    preds = train_transition_fold(train_texts, train_labels, test_by_node,
                                  LEARNING_RATE, NUM_EPOCHS, BATCH_SIZE, MAX_LENGTH)
    for n in NODES:
        for pid, p in zip(te_ids, preds[n['key']]):
            predB[n['key']][pid] = p
    print(f'  B fold{i} done')
print('モデルB 完了')

# 9. 結果①：ノード別 accuracy / precision / recall / F1（A vs B）

In [ ]:
def scores(pred_for_node, key):
    yt = [true_node[key][p] for p in patients]
    yp = [pred_for_node[p] for p in patients]
    return (accuracy_score(yt, yp),
            precision_score(yt, yp, average='macro', zero_division=0),
            recall_score(yt, yp, average='macro', zero_division=0),
            f1_score(yt, yp, average='macro', zero_division=0))


rows = []
for n in NODES:
    k = n['key']
    aA, pA, rA, fA = scores(predA[k], k)
    aB, pB, rB, fB = scores(predB[k], k)
    rows.append({'ノード': k,
                 'A acc': f'{aA:.3f}', 'B acc': f'{aB:.3f}',
                 'A F1': f'{fA:.3f}', 'B F1': f'{fB:.3f}',
                 'A prec': f'{pA:.3f}', 'B prec': f'{pB:.3f}',
                 'A rec': f'{rA:.3f}', 'B rec': f'{rB:.3f}'})
node_table = pd.DataFrame(rows)
print('===== ノード別 (A=単体painful / B=遷移) =====')
display(node_table)
node_table.to_csv(os.path.join(OUT_DIR, 'verify03_node.csv'), index=False, encoding='utf-8-sig')

# 10. 結果②：最終トリアージ（決定木を辿る）A vs B

In [ ]:
def triage_scores(pred_by_key):
    yt, yp = [], []
    for p in patients:
        by_bid = {n['bid']: pred_by_key[n['key']][p] for n in NODES}
        yp.append(predict_triage(by_bid))
        yt.append(true_triage[p])
    return accuracy_score(yt, yp), f1_score(yt, yp, average='macro', zero_division=0)


# サニティ：正解ノードラベルで辿ると真トリアージに一致するはず
_g = {n['key']: true_node[n['key']] for n in NODES}
print(f'[sanity] 正解ラベルで辿ったトリアージ一致率 = {triage_scores(_g)[0]:.3f}（1.000ならOK）')

tri_rows = []
for name, pred in [('A 単体(痛み/しびれ/振る舞い)', predA), ('B 遷移', predB)]:
    a, f = triage_scores(pred)
    tri_rows.append({'モデル': name, 'トリアージ acc': f'{a:.3f}', 'トリアージ macro-F1': f'{f:.3f}'})
tri_table = pd.DataFrame(tri_rows)
print('===== 最終トリアージ R3/R2/Y2 =====')
display(tri_table)
tri_table.to_csv(os.path.join(OUT_DIR, 'verify03_triage.csv'), index=False, encoding='utf-8-sig')
print('saved: verify03_node.csv, verify03_triage.csv')

# 11. まとめ

- **トリアージ/遷移の関数（branch_table, train_transition_fold, predict_triage）以外は painful のコードそのまま**。
- 比較は A（各ノード単体・162例×3）vs B（遷移・共有486例）の2つだけ。
- 見るのは ①ノード別 acc/prec/rec/F1（painfulと同じ指標）、②決定木を辿った最終トリアージ。
- fold・採用ペア・ハイパラは A/B 共通。